# Train the APR obstacle detector

This notebook trains the YOLO11n model that `app.py` uses to detect obstacles (windows, doors, AC units, meter panels, pipes, grills) on real walls. Run it in **Google Colab with a GPU runtime**: `Runtime -> Change runtime type -> T4 GPU`.

It walks through the same steps as `training/README.md` in the repo, in order:

1. Collect and label your own photos (AC units, meter panels, pipes, grills -- nothing public covers these)
2. Download the public door/window datasets
3. Merge everything into one dataset
4. Train YOLO11n (two stages: public data, then your photos)
5. Evaluate against the held-out test set
6. Download the exported `.onnx` file so you can commit it to the repo

**Before you start training (Section 4), you need labeled photos from Section 1 uploaded into this Colab session.** Sections 1-2 are things you do outside Colab (in Roboflow, in a browser) -- come back to this notebook once you have exported datasets ready to upload.

## 0. Check you have a GPU

If this prints "NVIDIA-SMI has failed", go to `Runtime -> Change runtime type` and pick a GPU (T4 is fine and free-tier), then re-run this cell.

In [ ]:
!nvidia-smi

## 1. Collect and label your own photos

**This part happens in your browser, not in Colab.** You're building the `team` dataset -- the one that covers `ac_unit`, `meter_panel`, `pipe` and `grill`, since no public dataset has these.

### 1.1 Take photos

- Aim for **40 to 80 labelled instances per class** (more is better; fewer than ~30 will struggle to generalize).
- Vary lighting (morning, noon, overcast, shade), camera distance, angle, wall colour/texture. Include clutter -- wires, plants, signs, shadows -- since real walls have it.
- Include some photos of clean walls with **no** obstacles at all -- the model needs negative examples too, otherwise it learns to always predict something.
- Shoot the same wall from a few angles/distances rather than only one photo per obstacle -- it multiplies your labeled instances without multiplying how many physical objects you need to find.

### 1.2 Label them in Roboflow (free tier is enough)

1. Go to [roboflow.com](https://roboflow.com), sign up free, create a new **Object Detection** project.
2. Upload your photos.
3. Draw a bounding box around every obstacle and label it with **exactly** one of these class names (case matters, must match exactly): `window`, `door`, `ac_unit`, `meter_panel`, `pipe`, `grill`.
   - If a wall has no obstacles, upload it anyway with zero boxes -- Roboflow keeps it as a "null" / clean example, which is what you want.
4. Once labeled, click **Generate** a new dataset version. Use a **70/10/20 train/valid/test split** (Roboflow's default splitter is fine). The 20% test split is your held-out set -- it must never be trained on, so don't hand-pick it to look good.
5. Click **Export**, choose format **"YOLOv8"** (this is the same as "YOLO" -- Roboflow relabels the format name sometimes, any YOLOv5/v8/v11-style plain YOLO txt-label export works, just avoid the OBB/segmentation-only variants), and choose **"zip download"** (not the code snippet).
6. This downloads a `.zip` to your computer. Keep it -- you'll upload it to Colab in the next cell.

### 1.3 Upload your export to this Colab session

Run the cell below, then pick the `.zip` file you just downloaded from Roboflow when the file picker appears.

In [ ]:
from google.colab import files
import zipfile, os

print("Select your Roboflow 'team' export .zip:")
uploaded = files.upload()
zip_name = next(iter(uploaded))

dest = "training/data/sources/team"
os.makedirs(dest, exist_ok=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(dest)

print(f"Extracted to {dest}:")
!find {dest} -maxdepth 2

Check the output above: you should see `data.yaml` plus `train/`, `valid/` and `test/` folders directly under `training/data/sources/team/`. If Roboflow nested everything inside one extra folder (e.g. `team/My-Project-1/data.yaml`), move the contents up one level before continuing:
```python
# only run this if you see an extra nested folder above
import shutil, glob
nested = glob.glob(f"{dest}/*/data.yaml")
if nested:
    inner = os.path.dirname(nested[0])
    for item in os.listdir(inner):
        shutil.move(os.path.join(inner, item), dest)
    shutil.rmtree(inner)
```

## 2. Download the public datasets (windows and doors)

These cover `window` and `door` so you don't have to photograph and label hundreds of those yourself. **Check each dataset's licence on its Roboflow Universe page before using it** -- the notebook doesn't do this for you.

- [Door and Window Detection](https://universe.roboflow.com/construction-plan/door-and-window-detection-fmw85) (5,382 images)
- [yolo-obb-1 wall/door/window](https://universe.roboflow.com/test-v0q9r/yolo-obb-1) (997 images, oriented boxes -- `build_dataset.py` converts these to axis-aligned boxes automatically)

### Option A -- Roboflow API (faster, needs a free API key)

Get your API key from Roboflow: click your profile icon -> **Settings** -> **API Keys** -> copy the "Private API Key". Paste it below (it's only used in this Colab session, not saved anywhere).

In [ ]:
ROBOFLOW_API_KEY = ""  # paste your key between the quotes, then run this cell

if ROBOFLOW_API_KEY:
    !pip install -q roboflow
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)

    # Door and Window Detection -> training/data/sources/door_window_detection/
    proj1 = rf.workspace("construction-plan").project("door-and-window-detection-fmw85")
    proj1.version(proj1.versions()[-1].version).download("yolov8", location="training/data/sources/door_window_detection")

    # yolo-obb-1 wall/door/window -> training/data/sources/wall_door_window_obb/
    proj2 = rf.workspace("test-v0q9r").project("yolo-obb-1")
    proj2.version(proj2.versions()[-1].version).download("yolov8", location="training/data/sources/wall_door_window_obb")

    print("Downloaded both public datasets via the Roboflow API.")
else:
    print("No API key set -- skip this cell and use Option B below instead.")

### Option B -- manual download (no API key needed)

If you'd rather not create an API key: open each dataset link above in your browser, click **Download this Dataset**, choose format **"YOLOv8"**, download the zip, then run the cell below twice (once per dataset) and pick the matching zip file each time.

In [ ]:
from google.colab import files
import zipfile, os

print("Select the 'Door and Window Detection' export .zip:")
uploaded = files.upload()
dest = "training/data/sources/door_window_detection"
os.makedirs(dest, exist_ok=True)
with zipfile.ZipFile(next(iter(uploaded))) as zf:
    zf.extractall(dest)
print(f"Extracted to {dest}")

In [ ]:
from google.colab import files
import zipfile, os

print("Select the 'yolo-obb-1 wall/door/window' export .zip:")
uploaded = files.upload()
dest = "training/data/sources/wall_door_window_obb"
os.makedirs(dest, exist_ok=True)
with zipfile.ZipFile(next(iter(uploaded))) as zf:
    zf.extractall(dest)
print(f"Extracted to {dest}")

## 3. Clone the repo and install dependencies

This clones `main` (which already has `training/build_dataset.py`, `training/train.py`, `training/evaluate.py` and `training/class_map.json`), then installs `ultralytics` and `onnx` -- **training-only** dependencies that never go into the app's `requirements.txt`/`pyproject.toml`.

Ultralytics (and weights trained with it) are **AGPL-3.0** licensed. That's fine while you're testing/evaluating. Before any commercial use, either buy an Ultralytics enterprise licence or retrain with a permissively licensed detector.

In [ ]:
!git clone https://github.com/JITESH-KUMAR05/APR-Sim-mini.git repo
%cd repo
!pip install -q ultralytics onnx

**If you already ran Sections 1-2 in this same Colab session before cloning**, the folders you built under `training/data/sources/...` are outside the freshly cloned `repo/` directory. Move them in:
```python
import shutil
shutil.move("training/data/sources", "repo/training/data/sources")
```
Otherwise (if you're running Sections 1-2 fresh from inside `repo/` after cloning), skip this -- you're already in the right place. To be safe, run the folder check below either way.

In [ ]:
# Sanity check: you should see team/, door_window_detection/ and wall_door_window_obb/,
# each containing a data.yaml plus train/valid/test folders.
!find training/data/sources -maxdepth 2 2>/dev/null || echo "Nothing here yet -- go back and run Sections 1-2, or move the folders in as shown above."

## 4. Check the class map

`training/class_map.json` says how each dataset's class names map onto APR's six classes. Open each dataset's `data.yaml` (printed below) and confirm the names match the map's keys. If a dataset names something differently (e.g. `Door` instead of `door`), edit `training/class_map.json` before continuing -- `build_dataset.py` prints any class it drops, so a mismatch is visible either way, but it's easy to miss.

In [ ]:
import json
print(json.dumps(json.load(open("training/class_map.json")), indent=2))
print()
for name in ("team", "door_window_detection", "wall_door_window_obb"):
    path = f"training/data/sources/{name}/data.yaml"
    print(f"--- {path} ---")
    try:
        print(open(path).read())
    except FileNotFoundError:
        print("(not found -- did you extract this dataset above?)")

## 5. Build the merged dataset

Public datasets feed `train`/`val` (their `test` splits go to `train` too -- they're not held out). Only `team`'s `test` split becomes the merged held-out test set, since that's the one you control and trust. If a class map entry doesn't exist for a folder, that folder is skipped -- check the printed output for anything unexpected.

In [ ]:
!python training/build_dataset.py

If it printed `WARNING: no held-out test images`, your `team` export didn't have a `test/` split (or Section 1 skipped it) -- go back and re-export from Roboflow with a real train/valid/test split before trusting any metric from Section 7.

## 6. Train

Two stages, as `training/README.md` describes:

- **Stage 1** starts from the stock `yolo11n.pt` pretrained weights and trains on everything in the merged set (public data + your photos, if you included `team` in Section 1). You can skip straight to a single-stage run if you don't have your own photos yet and just want to sanity-check the pipeline on public data alone -- but then only `window`/`door` will actually work well, since nothing else is in the public sets.
- **Stage 2** (optional, do this once you *do* have `team` photos in the merge) continues from Stage 1's best weights, letting the model specialize further on your photos.

Default is 100 epochs -- adjust `--epochs` if you're short on Colab GPU time (a free-tier session can disconnect after a few hours). Ultralytics saves checkpoints as it goes, so a shorter run still produces a usable (if less accurate) model.

In [ ]:
# Stage 1
!python training/train.py --epochs 100

In [ ]:
# Stage 2 -- only run this if 'team' (your own photos) is part of the merged dataset
!python training/train.py --weights training/runs/apr_obstacles/weights/best.pt --name apr_stage2 --epochs 50

Each run prints **per-class mAP50 on the held-out test split** at the end, and writes `models/apr_obstacles.onnx` -- overwritten each time you run `train.py`, so the last stage you ran is what ends up in `models/`.

## 7. Evaluate with the app's own detection pipeline

This runs the exported `.onnx` through the *exact* code `app.py` uses at inference time (letterboxing, NMS, thresholds) -- not just Ultralytics' own validation metrics -- so the numbers here are what you'd actually see in the app.

In [ ]:
!python training/evaluate.py --model models/apr_obstacles.onnx --images training/data/merged/test/images

Targets, from `training/README.md`:

- `window` and `door`: recall >= 0.90, precision >= 0.85 (the script above exits non-zero if either is missed)
- `ac_unit`, `meter_panel`, `pipe`, `grill`: mAP50 >= 0.70 (from Section 6's training output, not this script)
- Nothing in a test photo should get "no box at all" for windows and doors

If a class falls short, the usual fix is more/better-varied photos for that class, not more epochs.

## 8. Download the model and ship it

Run the cell below to download `apr_obstacles.onnx` to your computer. Then, **on your own machine** (not in Colab):

```bash
cp ~/Downloads/apr_obstacles.onnx models/apr_obstacles.onnx   # into your local clone of the repo
git add models/apr_obstacles.onnx
git commit -m "Add trained YOLO11n obstacle detector"
git push
```

Pushing triggers a Vercel redeploy. After it finishes, open the deployed app and confirm the "Trained model not loaded" banner is gone -- that's your signal the model is actually being used, not just present in the repo.

In [ ]:
from google.colab import files
files.download("models/apr_obstacles.onnx")